# Notebook 2: Data cleaning

This notebook cleans the merged dataset from notebook 1: it standardises numeric parsing (including comma-decimal columns) and removes administrative/merge-only metadata.

### Input
- `datasets/pre-processing/merged_crime_nbh_2024.csv`

### Output
- `datasets/pre-processing/cleaned_crime_nbh_2024.csv`


## Load merged dataset and packages

In [28]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

PREPROC_DIR = Path("datasets/pre-processing")
PREPROC_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    PREPROC_DIR / "merged_crime_nbh_2024.csv",
    na_values=["."],
    skipinitialspace=True,
    low_memory=False,
    dtype={
        "gwb_code_10": "string",
        "gwb_code_8": "string",
        "gwb_code": "string",
        "merge_key": "string",
        "ID": "string",
    },
)

df.shape

(14494, 133)

### Checking column types

In [29]:
# Columns that use comma as decimal separator (e.g. 7,3 → 7.3)
comma_decimal_cols = ['g_afs_hp', 'g_afs_gs', 'g_afs_kv', 'g_afs_sc', 'g_3km_sc']
for col in comma_decimal_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')

In [30]:
# Per column: count of cells that do not contain a number, and count of cells that contain '.'
stats = []
for col in df.columns:
    s = df[col]
    # Non-numeric: coerce to numeric, count NaN (includes empty, text, etc.)
    non_numeric = pd.to_numeric(s, errors='coerce').isna()
    # Exclude pandas NA from "was already missing" if you only want truly non-numeric content;
    # here we count all cells that are not a number
    n_non_numeric = non_numeric.sum()
    # Cells that contain the character '.'
    n_contains_dot = s.astype(str).str.contains(r'\.', regex=True, na=False).sum()
    stats.append({'column': col, 'non_numeric': n_non_numeric, 'contains_dot': n_contains_dot})

stats_df = pd.DataFrame(stats)
stats_df

,column,non_numeric,contains_dot
0,gwb_code_10,14494,0
1,gwb_code_8,517,0
2,regio,14494,238
3,gm_naam,14494,52
4,recs,14494,0
...,...,...,...
128,SoortMisdrijf,14494,14494
129,Perioden,14494,0
130,GeregistreerdeMisdrijven_1,0,14494
131,Gemeentenaam_2,14494,0


In [31]:
df.head()

,gwb_code_10,gwb_code_8,regio,gm_naam,recs,gwb_code,ind_wbi,a_inw,a_man,a_vrouw,a_00_14,a_15_24,a_25_44,a_45_64,a_65_oo,a_ongeh,a_gehuwd,a_gesch,a_verwed,a_nl_all,a_eur_al,a_neu_al,a_geb_nl,a_geb_eu,a_geb_ne,a_gbl_eu,a_gbl_ne,a_geb,p_geb,a_ste,p_ste,a_hh,a_1p_hh,a_hh_z_k,a_hh_m_k,g_hhgro,bev_dich,a_woning,a_nb_won,a_vastg,a_nb_vastg,g_wozbag,p_1gezw,p_1gezw_tw,p_1gezw_hw,p_1gezw_2w,p_1gezw_hvw,p_mgezw,p_leegsw,p_koopw,p_huurw,p_wcorpw,p_ov_hw,p_bj_me10,p_bj_mi10,g_ele,g_ele_tr,g_gas,p_stadsv,p_won_z_ag,p_won_m_ag,p_won_zs,p_won_ev,a_lp_pub,a_ons_po,a_ons_vovavo,a_ons_mbo,a_ons_hbo,a_ons_wo,a_opl_bvm,a_opl_hvm,a_opl_hw,a_arb_wz,p_arb_pp,p_arb_wn,p_arb_wnv,p_arb_wnf,p_arb_zs,a_inkont,g_ink_po,g_ink_pi,p_ink_li,p_ink_hi,g_hh_sti,p_hh_li,p_hh_hi,p_hh_lkk,p_hh_osm,p_hh_110,p_hh_120,m_hh_ver,a_soz_wb,a_soz_ao,a_soz_ww,a_soz_ow,a_jz_tn,p_jz_tn,a_wmo_t,p_wmo_t,a_bedv,a_bed_a,a_bed_bf,a_bed_gi,a_bed_hj,a_bed_kl,a_bed_mn,a_bed_oq,a_bed_ru,a_pau,a_bst_b,a_bst_nb,g_pau_hh,g_pau_km,a_m2w,g_afs_hp,g_afs_gs,g_afs_kv,g_afs_sc,g_3km_sc,a_opp_ha,a_lan_ha,a_wat_ha,pst_mvp,pst_dekp,ste_mvs,ste_oad,merge_key,ID,SoortMisdrijf,Perioden,GeregistreerdeMisdrijven_1,Gemeentenaam_2,SoortRegio_3
0,BU00140000,00140000,Binnenstad-Noord,Groningen,Buurt,BU00140000,1,4745,2470,2270,55,2365,1540,455,330,4190,330,180,40,3180,845,720,3180,150,230,695,490,10,3.0,15,4.0,3910,3255,565,85,1.2,12788.0,2532,0,583,0,297.0,12.0,11.0,1.0,0.0,0.0,88.0,12.0,13.0,87.0,19.0,68.0,94.0,6.0,1940.0,80.0,790.0,NaN,7.0,87.0,6.0,7.0,NaN,20,40,50,660,1520,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,120.0,90.0,60.0,290.0,NaN,NaN,NaN,NaN,1105,5.0,50.0,355.0,85.0,55.0,235.0,140.0,190.0,730,520,210,0.2,1971.0,65,0.4,0.3,0.6,0.5,16.1,39,37,2,9712.0,1.0,1.0,7001.0,BU00140000,83251,0.0.0,2024JJ00,1348.0,NaN,NaN
1,BU00140001,00140001,Binnenstad-Zuid,Groningen,Buurt,BU00140001,1,6975,3640,3340,110,3515,2220,625,515,6140,490,275,70,4350,1430,1200,4355,195,390,1230,810,15,2.0,15,2.0,5715,4725,865,125,1.2,12663.0,4017,2,920,0,275.0,8.0,6.0,1.0,0.0,0.0,92.0,11.0,12.0,88.0,24.0,64.0,96.0,4.0,1780.0,70.0,740.0,NaN,5.0,84.0,6.0,6.0,NaN,60,50,70,1060,2200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,220.0,130.0,60.0,460.0,NaN,NaN,NaN,NaN,1865,0.0,65.0,560.0,145.0,90.0,385.0,185.0,435.0,985,735,250,0.2,1788.0,100,0.3,0.3,0.6,0.8,14.4,59,55,4,9711.0,1.0,1.0,6698.0,BU00140001,83264,0.0.0,2024JJ00,2104.0,NaN,NaN
2,BU00140002,00140002,Binnenstad-Oost,Groningen,Buurt,BU00140002,1,4400,2275,2125,120,1815,1620,455,385,3795,365,190,55,2730,750,920,2730,125,240,625,680,15,4.0,10,3.0,3510,2815,560,130,1.3,16413.0,2740,0,212,0,252.0,13.0,11.0,2.0,0.0,0.0,87.0,6.0,11.0,88.0,32.0,56.0,88.0,12.0,1650.0,120.0,650.0,NaN,6.0,90.0,12.0,5.0,NaN,50,50,70,570,1140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.0,110.0,50.0,340.0,NaN,NaN,NaN,NaN,690,0.0,45.0,135.0,60.0,15.0,180.0,115.0,130.0,745,585,155,0.2,2771.0,60,0.5,0.5,0.5,0.8,14.8,29,27,2,9711.0,3.0,1.0,6355.0,BU00140002,83277,0.0.0,2024JJ00,295.0,NaN,NaN
3,BU00140003,00140003,Binnenstad-West,Groningen,Buurt,BU00140003,1,1740,935,800,25,750,645,160,155,1495,150,55,35,1200,305,235,1200,50,95,255,140,5,3.0,10,5.0,1410,1140,235,35,1.2,17792.0,1060,0,110,0,276.0,11.0,9.0,1.0,0.0,0.0,89.0,9.0,13.0,87.0,15.0,71.0,98.0,2.0,1790.0,50.0,730.0,NaN,14.0,81.0,4.0,15.0,NaN,10,10,10,240,440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,40.0,10.0,140.0,NaN,NaN,NaN,NaN,310,0.0,15.0,60.0,35.0,15.0,80.0,40.0,60.0,335,270,65,0.2,3442.0,40,0.6,0.3,0.6,1.0,13.9,11,10,1,9718.0,1.0,1.0,6700.0,BU00140003,83290,0.0.0,2024JJ00,352.0,NaN,NaN
4,BU00140004,00140004,Noorderplantsoen,Groningen,Buurt,BU00140004,1,10,5,5,0,0,0,10,0,10,5,0,0,5,0,0,10,5,0,0,0,0,0.0,0,0.0,5,0,5,0,2.0,52.0,0,0,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [32]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

In [33]:
print("Unique values in g_afs_hp column:", df["g_afs_hp"].unique())

Unique values in g_afs_hp column: [ 0.4  0.3  0.5  0.6  nan  0.7  1.   0.9  1.9  1.2  0.8  2.4  3.   2.6
  1.4  3.8  1.5  1.8  1.1  2.2  2.7  3.1  1.6  2.3  1.3  2.5  2.1  3.3
  2.   4.4  3.2  3.5  4.2  5.   5.2  4.   1.7  3.4  4.8  2.8  5.7  2.9
  5.1  5.9  7.7 10.5  3.9  4.5  5.4  3.7  7.2  9.2  6.   6.5  4.3  4.1
  4.9  4.6  7.1  6.2  3.6  5.3  7.   4.7  7.6  6.8  7.5  6.9  6.1  5.5
  6.4  6.7  7.3  9.7  7.4  8.5  6.6  0.2  6.3  5.8  5.6  8.6  0.1  0.
  9.1  9.8  8.3 11.6  7.9 10.4 10.3  9.   9.3  8.1  8.9  7.8  8.7  8. ]


In [34]:
# Identify object columns that could be numeric (used for sanity checks).
possible_numeric_obj_cols = []
for col in df.select_dtypes(include="object").columns:
    ser = df[col].astype(str).str.strip().str.replace(",", "", regex=False)
    is_numeric_like = ser.replace(".", "0", regex=False).str.replace("-", "", regex=False).str.isnumeric()
    fraction_numeric_like = is_numeric_like.mean()
    if fraction_numeric_like > 0.7:
        possible_numeric_obj_cols.append(col)
potential_num_cols = []

print(potential_num_cols)

[]


### Convert comma in decimal to dot

In [35]:
decimal_cols = ['g_afs_hp', 'g_afs_gs', 'g_afs_kv', 'g_afs_sc', 'g_3km_sc']

for col in decimal_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )

for col in decimal_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[decimal_cols].dtypes

g_afs_hp    float64
g_afs_gs    float64
g_afs_kv    float64
g_afs_sc    float64
g_3km_sc    float64
dtype: object

### Drop administrative / merge-only columns

In [36]:
df = df.drop(columns=["gwb_code_8", "recs", "ind_wbi", "gwb_code", "merge_key", "ID", "SoortMisdrijf", "Perioden", "Gemeentenaam_2", "SoortRegio_3"])

df.head()

,gwb_code_10,regio,gm_naam,a_inw,a_man,a_vrouw,a_00_14,a_15_24,a_25_44,a_45_64,a_65_oo,a_ongeh,a_gehuwd,a_gesch,a_verwed,a_nl_all,a_eur_al,a_neu_al,a_geb_nl,a_geb_eu,a_geb_ne,a_gbl_eu,a_gbl_ne,a_geb,p_geb,a_ste,p_ste,a_hh,a_1p_hh,a_hh_z_k,a_hh_m_k,g_hhgro,bev_dich,a_woning,a_nb_won,a_vastg,a_nb_vastg,g_wozbag,p_1gezw,p_1gezw_tw,p_1gezw_hw,p_1gezw_2w,p_1gezw_hvw,p_mgezw,p_leegsw,p_koopw,p_huurw,p_wcorpw,p_ov_hw,p_bj_me10,p_bj_mi10,g_ele,g_ele_tr,g_gas,p_stadsv,p_won_z_ag,p_won_m_ag,p_won_zs,p_won_ev,a_lp_pub,a_ons_po,a_ons_vovavo,a_ons_mbo,a_ons_hbo,a_ons_wo,a_opl_bvm,a_opl_hvm,a_opl_hw,a_arb_wz,p_arb_pp,p_arb_wn,p_arb_wnv,p_arb_wnf,p_arb_zs,a_inkont,g_ink_po,g_ink_pi,p_ink_li,p_ink_hi,g_hh_sti,p_hh_li,p_hh_hi,p_hh_lkk,p_hh_osm,p_hh_110,p_hh_120,m_hh_ver,a_soz_wb,a_soz_ao,a_soz_ww,a_soz_ow,a_jz_tn,p_jz_tn,a_wmo_t,p_wmo_t,a_bedv,a_bed_a,a_bed_bf,a_bed_gi,a_bed_hj,a_bed_kl,a_bed_mn,a_bed_oq,a_bed_ru,a_pau,a_bst_b,a_bst_nb,g_pau_hh,g_pau_km,a_m2w,g_afs_hp,g_afs_gs,g_afs_kv,g_afs_sc,g_3km_sc,a_opp_ha,a_lan_ha,a_wat_ha,pst_mvp,pst_dekp,ste_mvs,ste_oad,GeregistreerdeMisdrijven_1
0,BU00140000,Binnenstad-Noord,Groningen,4745,2470,2270,55,2365,1540,455,330,4190,330,180,40,3180,845,720,3180,150,230,695,490,10,3.0,15,4.0,3910,3255,565,85,1.2,12788.0,2532,0,583,0,297.0,12.0,11.0,1.0,0.0,0.0,88.0,12.0,13.0,87.0,19.0,68.0,94.0,6.0,1940.0,80.0,790.0,NaN,7.0,87.0,6.0,7.0,NaN,20,40,50,660,1520,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,120.0,90.0,60.0,290.0,NaN,NaN,NaN,NaN,1105,5.0,50.0,355.0,85.0,55.0,235.0,140.0,190.0,730,520,210,0.2,1971.0,65,0.4,0.3,0.6,0.5,16.1,39,37,2,9712.0,1.0,1.0,7001.0,1348.0
1,BU00140001,Binnenstad-Zuid,Groningen,6975,3640,3340,110,3515,2220,625,515,6140,490,275,70,4350,1430,1200,4355,195,390,1230,810,15,2.0,15,2.0,5715,4725,865,125,1.2,12663.0,4017,2,920,0,275.0,8.0,6.0,1.0,0.0,0.0,92.0,11.0,12.0,88.0,24.0,64.0,96.0,4.0,1780.0,70.0,740.0,NaN,5.0,84.0,6.0,6.0,NaN,60,50,70,1060,2200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,220.0,130.0,60.0,460.0,NaN,NaN,NaN,NaN,1865,0.0,65.0,560.0,145.0,90.0,385.0,185.0,435.0,985,735,250,0.2,1788.0,100,0.3,0.3,0.6,0.8,14.4,59,55,4,9711.0,1.0,1.0,6698.0,2104.0
2,BU00140002,Binnenstad-Oost,Groningen,4400,2275,2125,120,1815,1620,455,385,3795,365,190,55,2730,750,920,2730,125,240,625,680,15,4.0,10,3.0,3510,2815,560,130,1.3,16413.0,2740,0,212,0,252.0,13.0,11.0,2.0,0.0,0.0,87.0,6.0,11.0,88.0,32.0,56.0,88.0,12.0,1650.0,120.0,650.0,NaN,6.0,90.0,12.0,5.0,NaN,50,50,70,570,1140,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,190.0,110.0,50.0,340.0,NaN,NaN,NaN,NaN,690,0.0,45.0,135.0,60.0,15.0,180.0,115.0,130.0,745,585,155,0.2,2771.0,60,0.5,0.5,0.5,0.8,14.8,29,27,2,9711.0,3.0,1.0,6355.0,295.0
3,BU00140003,Binnenstad-West,Groningen,1740,935,800,25,750,645,160,155,1495,150,55,35,1200,305,235,1200,50,95,255,140,5,3.0,10,5.0,1410,1140,235,35,1.2,17792.0,1060,0,110,0,276.0,11.0,9.0,1.0,0.0,0.0,89.0,9.0,13.0,87.0,15.0,71.0,98.0,2.0,1790.0,50.0,730.0,NaN,14.0,81.0,4.0,15.0,NaN,10,10,10,240,440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,40.0,10.0,140.0,NaN,NaN,NaN,NaN,310,0.0,15.0,60.0,35.0,15.0,80.0,40.0,60.0,335,270,65,0.2,3442.0,40,0.6,0.3,0.6,1.0,13.9,11,10,1,9718.0,1.0,1.0,6700.0,352.0
4,BU00140004,Noorderplantsoen,Groningen,10,5,5,0,0,0,10,0,10,5,0,0,5,0,0,10,5,0,0,0,0,0.0,0,0.0,5,0,5,0,2.0,52.0,0,0,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,0,0.5,0.5,0.3,0.4,16.0,20,19,0,9717.0,2.0,1.0,6600.0,53.0


### Export Cleaned Dataset

In [37]:
out_path = PREPROC_DIR / "cleaned_crime_nbh_2024.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)

## Outputs saved

This notebook writes the cleaned tabular dataset to:
- `datasets/pre-processing/cleaned_crime_nbh_2024.csv`

This output is the input for notebooks 3 and 4.